### Get clear Categories from data after cleaning (Such as checking for language)

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from sklearn.utils import resample
from collections import Counter
from tqdm import tqdm



In [2]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: False
CUDA version: 12.8
GPU: No GPU


In [4]:
df = pd.read_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_name.pkl.gz")
df = df[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]

In [ ]:

# ------------------------------------------------------------
# 0) VORAUSSETZUNGEN
# ------------------------------------------------------------

TOP_CATEGORIES = [
    "Auto & Motorrad",
    "Möbel & Wohnen",
    "Werkzeug & Baumarkt",
    "Elektronik & Computer",
    "Spielzeug & Baby",
    "Kleidung & Accessoires",
    "Kosmetik & Drogerie",
    "Lebensmittel & Getränke",
    "Gesundheit & Pflege",
    "Bücher, Filme & Musik",
    "Sport & Freizeit",
    "Haustier & Tierbedarf",
    "Bürobedarf",
    "Sonstige"
]
# Heuristische Keyword-Regeln mit direkter, eindeutiger Zuordnung
# (falls ein Wort aus der Liste vorkommt → sofortige Zuordnung)
HEURISTIC_RULES = {
    "Auto & Motorrad": [
        "pkw", "motorradhelm", "neureifen", "autoreifen",
        "sommerreifen", "winterreifen", "allwetterreifen",
        "felge", "reifen", "kfz", "auto", "scheibenwischer",
        "motor", "motorrad", "scooter", "moped", "autoteil",
        "kennzeichen", "dachbox", "dachträger", "wagenheber", "motoröl",
    ],
    "Elektronik & Computer": [
        "gardine", "haushaltsgerät", "technik", "telefon","laptop", "smartphone", "fernseher", "tv", 
        "monitor", "kamera", "router", "konsole", "kühlschrank", "herd", "ofen", "computer", "elektronik", 
        "pc", "handy", "notebook", "handy", "tablet", "smartwatch", "bildschirm", "objektiv", "kopfhörer", 
        "router", "netzwerk", "tastatur", "konsole", "playstation", "xbox", "nintendo", "grafikkarte", "multimedia",
        "trockner", "waschmaschine", "spülmaschine", "küchenkleingerät", "küchengerät", "elektrogerät", 
        "digitalcamera", "digitalkamera"
    ],
    "Möbel & Wohnen": [
        "matratze", "bett", "sofa", "kommode", "lampe", "beleuchtung", "vorhang", "kissen", "teppich",  "wohnzimmer", "schlafzimmer", "zimmer", "wohnen", "möbel", "bad", "tisch", "stuhl", "regal", "schrank", "couch",
        "möbelzubehör", "pfanne", "geschirr", "topf", "töpfe", "esszimmer", "grill", "bettwäsche", "gardine", "decke", "decken", "küchenstuhl", "küchentisch",
        "gartenmöbel", "deko"
    ],
    "Werkzeug & Baumarkt": [
        "werkstatt", "akkuschrauber", "bohrer", "schraube",
        "säge", "schweißgerät", "dübel", "mörtel", "leiter",
        "werkzeug", "baumarkt", "hammer", "zange", "elektrowerkzeug", "gartenwerkzeug", "farbe", "pinsel", "spachtel", "schraubenzieher", "sägen", "bohrmaschine",
        "gartengeräte", "rasenmäher", "heckenschere", "gartenschlauch"
    ],
    "Spielzeug & Baby": [
        "lego", "puppe", "spielzeug", "kinderwagen",
        "wickel", "kuscheltier", "baustein", "puzzle", "spiele", "spielware",
        "toys", "games", "wasserspielzeug", "spielware"
    ],
    "Kleidung & Accessoires": [
        "t-shirt", "hose", "jacke", "kleid", "schuh", "sneaker",
        "unterwäsche", "schmuck", "tasche", "gürtel", "mütze",
        "accessoire", "mode", "kleidung", "rock", "jeans", "uhr", "rucksack", "taschen", "körperpflege", "radbekleidung",
        "sportbekleidung", "anzug", "bluse", "sportschuh", "damenschuh", "herrenschuh", "kinderschuh",
        "rucksäcke", "shoes"
    ],
    "Kosmetik & Drogerie": [
        "parfum", "deo", "shampoo", "seife", "make-up", "nagellack", "makeup", "kosmetik", "drogerie", "dusche", "rasur", "spülung", "seife", "zahnpasta",
        "kontaktlinsen", "creme", "lotion", "beauty", "handpflege", "körperpflege", "foundation", "haare",
        "gesichtspflege"
    ],
    "Lebensmittel & Getränke": [
        "kaffee", "spirituose", "tee", "bier", "wein", "snack", "essen", "getränk", "lebensmittel",
        "nahrung", "brot", "whyski", "haushaltsgerät", "kaffee", "tee"
    ],
    "Gesundheit & Pflege": [
        "vitamin", "arzneimittel", "medikament", "vitamin", "verband", "pflaster", "desinfektion", "gesundheit", "pflege", "apotheke", "thermometer", "gesundheit",
        "gesichtspflege", "körperpflege"
    ],
    "Bücher, Filme & Musik": [
        "buch", "film", "dvd", "blu-ray", "cd", "hörbuch", "zeitschrift", "musik", "schallplatte", "hörbuch", "hörspiel", 
    ],
    "Sport & Freizeit": [
        "fahrrad", "helm", "hantel", "yoga", "zelt", "angeln", "ski", "trampolin", "sport", "fitness", "laufen", "joggen", "tennis", "fußball"
    ],
    "Haustier & Tierbedarf": [
        "hundefutter", "katzenstreu", "kratzbaum", "hundespielzeug", "aquarium", "nager", "tier", "haustier", "hund", "katze", "fisch", "vogel", "käfig", "terrarium", "leckerli"
    ],
    "Bürobedarf": [
        "ordner", "hefter", "locher", "druckerpapier", "kugelschreiber", "post-it", "drucker", "scanner", "bürobedarf", "papier", "büro", "stift", "kalender"
    ],
    "Sonstige": [
    ],
    "Undefiniert": [
    ]
}

# Pseudo-Kategorien, die oft als oberste Ebene auftauchen und übersprungen werden sollen
STOPWORDS_FIRST_LEVEL = {
    "damen", "herren", "kinder", "mädchen", "jungen", "baby", "babys",
    "sale", "angebote", "neuheiten", "marke", "brand", "top", "neu"
}

# ------------------------------------------------------------
# 1) HEURISTISCHE ZUORDNUNG
# ------------------------------------------------------------
def normalize_text(s: str) -> str:
    s = s.lower()
    # vereinheitliche Trennzeichen zu ">"
    s = re.sub(r"[|/:\\>]+", ">", s)
    # entferne Sonderzeichen, mehrfachspaces
    s = re.sub(r"[^0-9a-zäöüß><\s\-\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s
    
def tokenize(text: str):
    return set(text.lower().split())

def match_with_suffix(word, tokens):
    suffixes = ["", "e", "en", "n", "s", "er"]
    for suf in suffixes:
        if word + suf in tokens:
            return True
    return False
    
def extract_all_meaningful_segments(cat): 
    """ Gibt eine Liste aller Segmente zurück, überspringt dabei Stopwords. """ 
    if not isinstance(cat, str) or not cat.strip(): 
        return ["sonstige"] 
        
    s = normalize_text(cat) 
        
    parts = [p.strip() for p in s.split(">") if p.strip()] 
    if not parts: 
        return ["sonstige"] 

    return [p for p in parts if p not in STOPWORDS_FIRST_LEVEL] or [parts[0]]


def heuristic_match(cat):
    """
    Nimmt alle Segmente einer Kategorie, prüft gegen HEURISTIC_RULES.
    Gibt die Kategorie mit den meisten Treffern zurück.
    """
    segments = extract_all_meaningful_segments(cat)
    tokens = tokenize(" ".join(segments))
    
    counts = Counter()
    for cat_name, keys in HEURISTIC_RULES.items():
        for k in keys:
            if match_with_suffix(k, tokens):
                counts[cat_name] += 1
    
    if not counts:
        return "Sonstige"
    
    return counts.most_common(1)[0][0]

# ------------------------------------------------------------
# 2) HAUPTFUNKTION: MAPPING PIPELINE
# ------------------------------------------------------------
def map_shop_categories(df: pd.DataFrame, col: str = "shop_cat") -> pd.DataFrame:
    out = df.copy()
    tqdm.pandas(desc="Prozessiere Kategorien")
    out["top_category_mapped"] = out["shop_cat"].progress_apply(heuristic_match)
    
    return out

# ------------------------------------------------------------
# 3) REPORTING / EXPORT
# ------------------------------------------------------------
def summarize_and_export(out_df: pd.DataFrame, original_col: str = "shop_cat"):
    # Reduktionsübersicht
    n_original = out_df[original_col].nunique(dropna=True)
    n_mapped   = out_df["top_category_mapped"].nunique(dropna=True)

    print(f"Einzigartige Original-Kategorien: {n_original}")
    print(f"Einzigartige Zielkategorien (aus deinem Set): {n_mapped} von {len(TOP_CATEGORIES)} möglichen")
    print()
    print("Top 20 Mapped Kategorien (Count):")
    print(out_df["top_category_mapped"].value_counts())
    print(out_df.head())

    # Speichere Mapping (ein Datensatz je Zeile, inkl. Score & Quelle)
    mapping = (
        out_df[[original_col,"top_category_mapped"]] # "tfidf_score", ,"heuristic_hit"
        .copy()
    )
    mapping.to_csv("Categories/category_mapping.csv", index=False)

    # Häufigkeiten der Zielkategorien
    counts = out_df["top_category_mapped"].value_counts().rename_axis("top_category").reset_index(name="count")
    counts.to_csv("Categories/top_category_counts.csv", index=False)

    return mapping, counts

# ------------------------------------------------------------
# 4) BEISPIEL-DURCHLAUF (auskommentiert, damit du es selbst steuerst)
# ------------------------------------------------------------
out_df = map_shop_categories(df, col="shop_cat")
mapping, counts = summarize_and_export(out_df, original_col="shop_cat")


# ------------------------------------------------------------
# 5) In pickle umwandeln für spätere Sets
# ------------------------------------------------------------
out_df.to_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_title_only_mainentity_with_new_category.pkl.gz", compression="gzip")

Prozessiere Kategorien: 100%|██████████| 3030931/3030931 [13:32<00:00, 3731.51it/s]


Einzigartige Original-Kategorien: 161947
Einzigartige Zielkategorien (aus deinem Set): 14 von 14 möglichen

Top 20 Mapped Kategorien (Count):
top_category_mapped
Sonstige                   819195
Kleidung & Accessoires     622017
Elektronik & Computer      413625
Möbel & Wohnen             300628
Auto & Motorrad            287703
Werkzeug & Baumarkt        143577
Spielzeug & Baby           111343
Sport & Freizeit            99528
Gesundheit & Pflege         87400
Kosmetik & Drogerie         81104
Lebensmittel & Getränke     28169
Bürobedarf                  17884
Haustier & Tierbedarf       16146
Bücher, Filme & Musik        2612
Name: count, dtype: int64
             id  product_id  \
0    1817742672  2820098210   
1    4075272753   852480790   
4    2074416376  1217832594   
6    1527509196  1038150077   
8  156275941980  3621538713   

                                                name  \
0  Vipack: Hochbett / Etagenbett "BONNY" Weiß / B...   
1  Sommerreifen PIRELLI P-ZERO (NEW) 

A Manual check of the fisrt 200 categories were done to confirm that the items were corectly placed. All items were sorted correctly

Ignore the Rest for now

In [6]:
def improve_sonstige_with_name_and_description(df,
                                               name_col,
                                               desc_col) -> pd.DataFrame:
    out = df.copy()

    # 1. Nur Zeilen mit 'sonstige'
    mask = out["top_category_mapped"] == "Sonstige"
    df_sonstige = out[mask].copy()

    # 2. Kombiniere Name + Beschreibung zu einem Textfeld
    combined_text = (
        df_sonstige[name_col].fillna("") + " " + df_sonstige[desc_col].fillna("")
    ).str.strip()

    # 3. Heuristische Kategorisierung
    df_sonstige["heuristic_from_text"] = combined_text.apply(heuristic_match)

    # 4. TF-IDF (nur für die "sonstige"-Fälle)
    profiles = build_category_profiles(CATEGORY_SEEDS)
    tfidf_labels, tfidf_scores = tfidf_assign(combined_text, profiles)
    df_sonstige["tfidf_from_text"] = tfidf_labels
    df_sonstige["tfidf_score_from_text"] = tfidf_scores

    # 5. Beste neue Kategorie wählen (Heuristik > TF-IDF)
    df_sonstige["reassigned_category"] = df_sonstige["heuristic_from_text"].fillna(df_sonstige["tfidf_from_text"])

    # 6. Zurückschreiben in Original-DF
    out.loc[mask, "top_category_mapped"] = df_sonstige["reassigned_category"]

    return out


out_df = improve_sonstige_with_name_and_description(out_df,
                                                    name_col="name",
                                                    desc_col="desc")

n_mapped   = out_df["top_category_mapped"].nunique(dropna=True)

print(f"Einzigartige Zielkategorien (aus deinem Set): {n_mapped} von {len(TOP_CATEGORIES)} möglichen")
print()
print("Top 20 Mapped Kategorien (Count):")
print(out_df["top_category_mapped"].value_counts().head(20))

Einzigartige Zielkategorien (aus deinem Set): 14 von 14 möglichen

Top 20 Mapped Kategorien (Count):
top_category_mapped
Kleidung & Accessoires     652525
Sonstige                   500988
Möbel & Wohnen             351137
Auto & Motorrad            332132
Elektronik & Computer      277331
Werkzeug & Baumarkt        215929
Sport & Freizeit           152065
Spielzeug & Baby           131384
Kosmetik & Drogerie        117626
Gesundheit & Pflege        103595
Lebensmittel & Getränke     64594
Bürobedarf                  34628
Haustier & Tierbedarf       27254
Bücher, Filme & Musik       18602
Name: count, dtype: int64


In [7]:
def sample_products_per_category(
    df, n, cat_col="top_category_mapped", name_col="name", desc_col="desc",
    random=False, seed=None
):
    """
    Gibt pro Kategorie n Beispielprodukte mit Name + Description zurück.
    Wenn random=True, werden pro Kategorie zufällige n Zeilen gezogen.
    """
    cols = [cat_col, name_col, desc_col]
    g = df.loc[:, cols].groupby(cat_col, group_keys=False)

    if random:
        # directly sample per group, no FutureWarning
        return g.sample(n=n, random_state=seed).reset_index(drop=True)
    else:
        return g.head(n).reset_index(drop=True)


sample_df = sample_products_per_category(out_df, n=5)              # first 5 per category
sample_df_rand = sample_products_per_category(out_df, n=5, random=True, seed=42)  # random 5 per category

print(sample_df)
print(sample_df_rand)


      top_category_mapped                                               name  \
0          Möbel & Wohnen  Vipack: Hochbett / Etagenbett "BONNY" Weiß / B...   
1         Auto & Motorrad  Sommerreifen PIRELLI P-ZERO (NEW) S.C.  215/45...   
2   Elektronik & Computer  Telekom Speedphone 51 Festnetztelefon (mit Bas...   
3     Werkzeug & Baumarkt  Spiegelschrank »Basic« 60 cm weiß, Möbelpartne...   
4     Werkzeug & Baumarkt  Gelenkarmmarkise SPETTMANN STAR Markisen Gr. 3...   
..                    ...                                                ...   
65  Bücher, Filme & Musik         Lacoste Game Advance Tennisschuh weiß 42,0   
66  Haustier & Tierbedarf  DeliBest Ochsenziemer Premium 15 Centimeter 6x...   
67             Bürobedarf  Maul Eclipse 8200295 LED-Schreibtischleuchte 7...   
68             Bürobedarf  HP Universal - Schweres Papier, matt - 125 Mik...   
69  Bücher, Filme & Musik  Schwanensee Ballett - Ballett erzählt als Hörs...   

                                       

In [ ]:
# 📌 Deine Zielkategorien
TOP_CATEGORIES = [
    "Auto & Motorrad", "Möbel & Wohnen", "Werkzeug & Baumarkt", "Elektronik & Computer",
    "Spielzeug & Baby", "Kleidung & Accessoires", "Kosmetik & Drogerie", "Lebensmittel & Getränke",
    "Gesundheit & Pflege", "Bücher, Filme & Musik", "Sport & Freizeit",
    "Haustier & Tierbedarf", "Bürobedarf", "Sonstige"
]

def train_bert_classifier(train_df, name_col="name", desc_col="desc", label_col="top_category_mapped"):
    # Combine name + description into one text field
    train_df["text"] = train_df[name_col].fillna("") + " " + train_df[desc_col].fillna("")

    # Encode labels
    le = LabelEncoder()
    le.fit(TOP_CATEGORIES)
    train_df["label"] = le.transform(train_df[label_col])

    # Convert to HuggingFace Dataset
    dataset = Dataset.from_pandas(train_df[["text", "label"]])

    # Tokenizer & Model
    tokenizer = BertTokenizer.from_pretrained("bert-base-german-cased")
    model = BertForSequenceClassification.from_pretrained("bert-base-german-cased", num_labels=len(le.classes_))

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True)

    tokenized_ds = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Training setup
    args = TrainingArguments(
        output_dir="bert-category-classifier",
        evaluation_strategy="no",
        per_device_train_batch_size=64,
        num_train_epochs=3,
        save_strategy="no",
        logging_steps=10,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    # Train!
    trainer.train()

    return model, tokenizer, le
def predict_categories(model, tokenizer, label_encoder, df_sonstige, name_col="name", desc_col="desc", batch_size=1024):
    df_sonstige["text"] = df_sonstige[name_col].fillna("") + " " + df_sonstige[desc_col].fillna("")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    predictions = []

    # Process in batches to avoid OOM
    texts = df_sonstige["text"].tolist()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        predictions.extend(preds)

    df_sonstige["bert_category"] = label_encoder.inverse_transform(predictions)
    return df_sonstige

# 1. Trainingsdaten = alle Produkte mit guter Kategorie (≠ "Sonstige") 
train_df = out_df[out_df["top_category_mapped"] != "Sonstige"] 

n_per_class = 10000 
sampled_dfs = []

for cat in train_df["top_category_mapped"].unique():
    group = train_df[train_df["top_category_mapped"] == cat]
    sampled = resample(
        group,
        replace=False,        # no duplicates
        n_samples=min(len(group), n_per_class),
        random_state=42
    )
    sampled_dfs.append(sampled)

def predict_categories_2_option(model, tokenizer, label_encoder, df_sonstige, 
                       name_col="name", desc_col="desc", shop_cat_col="shop_cat", 
                       batch_size=1024):
    """
    Zuerst wird versucht, shop_cat zu verwenden.
    Falls shop_cat nicht existiert oder kein sicheres Mapping möglich ist,
    wird die BERT-Vorhersage auf Basis von name+desc genutzt.
    """

    # Shop_cat soweit möglich direkt übernehmen
    def map_shop_cat(x):
        if pd.isna(x):  # echte NaN-Werte
            return None
        if isinstance(x, str) and x.strip() == "":  # leere Strings
            return None
        if x in TOP_CATEGORIES:  # passt direkt
            return x
        return None  # alles andere -> BERT

    df_sonstige["predicted_category"] = df_sonstige[shop_cat_col].apply(map_shop_cat)

    # Für alle, wo predicted_category noch None ist -> BERT verwenden
    mask_needs_bert = df_sonstige["predicted_category"].isna()

    if mask_needs_bert.any():
        df_sonstige.loc[mask_needs_bert, "text"] = (
            df_sonstige.loc[mask_needs_bert, name_col].fillna("") + " " +
            df_sonstige.loc[mask_needs_bert, desc_col].fillna("")
        )

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.eval()

        texts = df_sonstige.loc[mask_needs_bert, "text"].tolist()
        predictions = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model(**inputs)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            predictions.extend(preds)

        df_sonstige.loc[mask_needs_bert, "predicted_category"] = label_encoder.inverse_transform(predictions)

    # Ergebnis-Spalte heißt jetzt predicted_category
    return df_sonstige

train_df_sampled = pd.concat(sampled_dfs).reset_index(drop=True)

# 2. Trainiere BERT-Modell
model, tokenizer, label_encoder = train_bert_classifier(train_df_sampled)

# 3. Finde alle "Sonstige"-Produkte
df_sonstige = out_df[out_df["top_category_mapped"] == "Sonstige"].copy()

# 4. BERT-Vorhersage
df_sonstige = predict_categories_2_option(model, tokenizer, label_encoder, df_sonstige)

# 5. Schreibe neue Vorhersage zurück in Original-DataFrame
out_df.loc[df_sonstige.index, "top_category_mapped"] = df_sonstige["bert_category"]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/485k [00:00<?, ?B/s]

/work/kelagin/Entity-Matching-Pipeline-for-German-Product-Data---Master-Thesis/.venv/lib64/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/130000 [00:00<?, ? examples/s]

/work/kelagin/Entity-Matching-Pipeline-for-German-Product-Data---Master-Thesis/.venv/lib64/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss
10,2.543700
20,2.164700
30,1.848500
40,1.566900
50,1.370600
60,1.351100
70,1.210500
80,1.294500
90,1.191300
100,1.192300


In [10]:

df_sonstige.head(100)

out_df.to_csv("catgeories_with_Bert.csv")
out_df.to_pickle("../data/working/categories_with_Bert.pkl.gz")

In [11]:
out_df.to_pickle("../data/working/categories.pkl.gz")

In [12]:
out_df.to_csv("categories_bert.csv")

In [13]:
df_sonstige.head(10)

,shop_id,ean,mpnr,price,product_id,name,cat_id,id,brand,shop_cat,...,name+description+brand,has_long_name,is_mainentity,cat_first,heuristic_hit,tfidf_label,tfidf_score,top_category_mapped,text,bert_category
39,26124,4.242004e+12,NaN,569.00,3866257091,Neff N30 ECB1622i Elektroherd (E1CCD2AN1),98440,4820412964,Neff,Haushalt Kochen & Backen Einbauherde,...,neff n30 ecb1622i elektroherd (e1ccd2an1) neff...,True,True,haushalt kochen backen einbauherde,None,Sonstige,0.000000,Sonstige,Neff N30 ECB1622i Elektroherd (E1CCD2AN1) Neff...,Möbel & Wohnen
50,10992,4.242004e+12,T66BT6QN2,848.52,1677970320,Neff T66BT6QN2 N70 Autarkes Induktionskochfeld...,143003,1848041189,Neff,Haushaltsgeräte|Backöfen & Herde|Kochfelder,...,neff t66bt6qn2 n70 autarkes induktionskochfeld...,True,True,haushaltsgeräte,None,Sonstige,0.000000,Sonstige,Neff T66BT6QN2 N70 Autarkes Induktionskochfeld...,Möbel & Wohnen
51,3117,6.934433e+12,103972108S,215.10,1452129898,LS2 FF397 Vector HPFC Evo Kripton Integralhelm...,1705,3044852733,LS2,Helme > Integral-Helme,...,ls2 ff397 vector hpfc evo kripton integralhelm...,True,True,helme,None,Sonstige,0.000000,Sonstige,LS2 FF397 Vector HPFC Evo Kripton Integralhelm...,Sport & Freizeit
57,16880,1.941821e+11,774591,109.95,1790941216,New Balance CM 996 Sneaker Black 81⁄2,98546,1382180060,New Balance,Neu->Sportarten->Lifestyle->Freizeitschuhe,...,new balance cm 996 sneaker black 81⁄2 innovati...,True,True,neu-,None,Sonstige,0.000000,Sonstige,New Balance CM 996 Sneaker Black 81⁄2 Innovati...,Kleidung & Accessoires
71,24360,3.401381e+12,11027031,7.43,615688345,Bioderma Atoderm Intensive bei Neurodermitis,7161,3844916958,NAOS Deutschland GmbH,Bioderma Atoderm,...,bioderma atoderm intensive bei neurodermitis b...,True,True,bioderma atoderm,None,Sonstige,0.000000,Sonstige,Bioderma Atoderm Intensive bei Neurodermitis B...,Gesundheit & Pflege
100,15616,4.251166e+12,NaN,798.49,3765985841,Mafell Akku Tauchsäge MT 55 18M bl PURE T-MAX ...,142361,4607647674,Mafell,Produkte > Elektrowerkzeug > Akku-Säge > Akku-...,...,mafell akku tauchsäge mt 55 18m bl pure t-max ...,True,True,produkte,None,Sonstige,0.000000,Sonstige,Mafell Akku Tauchsäge MT 55 18M bl PURE T-MAX ...,Werkzeug & Baumarkt
115,25613,4.064033e+12,NaN,79.99,3651107794,"Gabor Keilsandalette, in Weite G (=weit)",98544,4623771505,GABOR,Damen/Damenschuhe,...,"gabor keilsandalette, in weite g (=weit) keils...",True,True,damenschuhe,None,Sonstige,0.000000,Sonstige,"Gabor Keilsandalette, in Weite G (=weit) Keils...",Kleidung & Accessoires
119,5440,3.326745e+12,3BN67360AA,1029.02,1825716652,Alcatel-Lucent 8262Ex DECT - Schnurloses Digit...,98946,2069961137,Alcatel,Telefone und Handys > Telefone divers,...,alcatel-lucent 8262ex dect - schnurloses digit...,True,True,telefone und handys,None,Sonstige,0.000000,Sonstige,Alcatel-Lucent 8262Ex DECT - Schnurloses Digit...,Elektronik & Computer
132,24376,7.613037e+12,16016138,19.93,1945738130,OPTIFAST home Suppe Kartoffel-Lauch,5770,4687784193,Nestle Health Science (Deutschland) GmbH,,...,optifast home suppe kartoffel-lauch optifast h...,True,True,sonstige,Sonstige,Sonstige,0.114985,Sonstige,OPTIFAST home Suppe Kartoffel-Lauch OPTIFAST h...,Möbel & Wohnen
164,24395,8.710104e+12,NaN,109.99,2837028401,"Dampfbügeleisen Philips Azur Pro GC4902/20 0,3...",142875,3878527188,Philips,"Heim > Haushalt > Wäschetrockner, Bügeleisen u...",...,"dampfbügeleisen philips azur pro gc4902/20 0,3...",True,True,heim,None,Sonstige,0.000000,Sonstige,"Dampfbügeleisen Philips Azur Pro GC4902/20 0,3...",Gesundheit & Pflege


In [3]:
category_overview = pd.read_pickle("../data/working/categories.pkl.gz")

print("Top 20 Mapped Kategorien (Count):")
print(category_overview["top_category_mapped"].value_counts().head(20))

Top 20 Mapped Kategorien (Count):
top_category_mapped
Kleidung & Accessoires     682898
Möbel & Wohnen             508220
Auto & Motorrad            348395
Elektronik & Computer      327227
Werkzeug & Baumarkt        304163
Sport & Freizeit           186920
Spielzeug & Baby           148176
Kosmetik & Drogerie        145777
Gesundheit & Pflege        125724
Lebensmittel & Getränke     82727
Bürobedarf                  59341
Haustier & Tierbedarf       34020
Bücher, Filme & Musik       26202
Name: count, dtype: int64
